Important : Use this file only to Train the model Not For Trail Running ( if you want to run the pre trained model use recommend.py )

In [3]:
import numpy as np
import pandas as pd

In [4]:
# Loading datasets
movies = pd.read_csv("./dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("./dataset/tmdb_5000_credits.csv")

Preprocessing - Start

In [5]:
# merging both movies and credits dataset into single dataset
movies = movies.merge(credits,on='title')
# display(movies.columns)

In [6]:
# important columns: genres, id, keywords, overview, tagline, title,vote_average,vote_count
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
# display(movies.head(1))

In [7]:
#droping null values
movies.dropna(inplace=True)

In [8]:
import ast
def Extract_Keywords_From_Json_Data(obj):
    keywords = []
    for i in ast.literal_eval(obj):
        keywords.append(i["name"])
    return keywords


def Extract_Keywords_From_Cast_Column(obj):
    keywords = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            keywords.append(i["name"])
            counter+1
        else:
            break
    return keywords 


def Extract_Keywords_From_Crew_Column(obj):
    keywords = []
    for i in ast.literal_eval(obj):
        if i["job"]=='Director':
            keywords.append(i["name"])
            break
    return keywords 

In [9]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies['genres'] = movies['genres'].apply(Extract_Keywords_From_Json_Data)
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(Extract_Keywords_From_Json_Data)
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(Extract_Keywords_From_Cast_Column)
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(Extract_Keywords_From_Crew_Column)
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])


In [10]:
# display(movies.head(5))

In [11]:
# adding new column "tags" to the movies DataFrame
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [12]:
new_df = movies[['movie_id','title','tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [13]:
# display(new_df)

NLTK(Natural Lang ToolKit) - Stemming ( ex : walking -> walk, walked -> walk, walk -> walk)

In [14]:
from nltk.stem.porter import PorterStemmer

# stemming converts words to their root form
ps = PorterStemmer()

def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))     #stem is used to convert loving -> love, loved -> love, love -> love
    return " ".join(y)

In [15]:
import joblib


# Stemming each words to their root form ( ex : loving -> love, loved -> love, love -> love)
new_df['tags'] = new_df['tags'].apply(stem)

# storing the new_df in the pkl file
joblib.dump(new_df,"DF.pkl")

['DF.pkl']

sklearn - CounterVector (To Convert text to numerical data)

In [16]:
# converting text to numerical data using word frequency
from sklearn.feature_extraction.text import CountVectorizer
import joblib
cv = CountVectorizer(max_features=5000, stop_words='english') # stop words removes the connecting words like (a,an,and,are etc) 

vectors = cv.fit_transform(new_df['tags']).toarray()


# storing the vectors in the pkl file
joblib.dump(vectors,"vectors.pkl") 

['vectors.pkl']

Preprocessing - End

Training Model - Start

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)

In [ ]:
recommend('Batman Begins')

In [ ]:
import joblib
from sklearn.metrics.pairwise import cosine_similarity

DF = joblib.load('DF.pkl')
vectors = joblib.load('vectors.pkl')

similarity = cosine_similarity(vectors)

def recommend(movie):
    movie_index = DF[DF['title'] == movie].index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)

In [21]:
recommend('Avatar')

Aliens vs Predator: Requiem
Battle: Los Angeles
Predator
Independence Day
Falcon Rising
